<a href="https://colab.research.google.com/github/flatironinstitute/2026-flatiron-cmb-summer-school/blob/main/notebooks/CMB_School_Power_Spectrum_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Power Spectrum Analysis

This notebook, featured in the 2026 Flatiron Institute CMB Summer School, is based on 
https://github.com/jeffmcm1977/CMBAnalysis_SummerSchool (credits: Jeff McMahon, Renee Hlozek, Sigurd Naess), with further revisions and contributions by Alex van Engelen, Alexandra Rahlin, Mathew Madhavacheril, Joshua Kim, Zach Atkins, Will Coulton, Melanie Archipley, Tom Crawford.

In this notebook, we will study power spectrum estimation, using maps that we made in previous courses. The goal of this notebook is to understand the multiple steps necessary to get a cosmological-ready power spectrum from a realistic sky map.

In [48]:
!python -c "import cmb_modules" || ( \
    wget https://github.com/jeffmcm1977/CMBAnalysis_SummerSchool/raw/master/cmb_school.tar.gz && \
    tar xzvf cmb_school.tar.gz \
)

In [49]:
import numpy as np
import matplotlib
import sys
import matplotlib.cm as cm
import matplotlib.mlab as mlab
import matplotlib.pyplot as plt
import astropy.io.fits as fits

import constants as cs # the constants module
import cmb_modules # the module of functions

N = cs.N
c_min = cs.c_min
c_max = cs.c_max
X_width =cs.X_width
Y_width = cs.Y_width
beam_size_fwhp = cs.beam_size_fwhp

pix_size = cs.pix_size

Number_of_Sources  = cs.Number_of_Sources
Amplitude_of_Sources = cs.Amplitude_of_Sources
Number_of_Sources_EX = cs.Number_of_Sources_EX
Amplitude_of_Sources_EX = cs.Amplitude_of_Sources_EX

Number_of_SZ_Clusters  = cs.Number_of_SZ_Clusters
Mean_Amplitude_of_SZ_Clusters = cs.Mean_Amplitude_of_SZ_Clusters
SZ_beta = cs.SZ_beta
SZ_Theta_core = cs.SZ_Theta_core

white_noise_level = cs.white_noise_level
atmospheric_noise_level = cs.atmospheric_noise_level
one_over_f_noise_level = cs.one_over_f_noise_level

## Recalculating the results from the previous stages

Let's start by making a realistic CMB map using what we've seen in previous courses.

In [ ]:
## Make a CMB map
ell, DlTT = np.loadtxt("CAMB_fiducial_cosmo_scalCls.dat", usecols=(0, 1), unpack=True) 
CMB_T = cmb_modules.make_CMB_T_map(N,pix_size,ell,DlTT)

## make a point source map
PSMap = cmb_modules.Poisson_source_component(N,pix_size,Number_of_Sources,Amplitude_of_Sources) 
PSMap +=  cmb_modules.Exponential_source_component(N,pix_size,Number_of_Sources_EX,Amplitude_of_Sources_EX)

## make an SZ map
SZMap,SZCat = cmb_modules.SZ_source_component(N,pix_size,Number_of_SZ_Clusters,Mean_Amplitude_of_SZ_Clusters,SZ_beta,SZ_Theta_core,False)

## add them all together to get the sky map at a single freuqency
total_map = CMB_T + PSMap + SZMap

## incorperate the impact of the instrument
    ## beam
CMB_T_convolved = cmb_modules.convolve_map_with_gaussian_beam(N,pix_size,beam_size_fwhp,total_map)
    ## noise
Noise = cmb_modules.make_noise_map(N,pix_size,white_noise_level,atmospheric_noise_level,one_over_f_noise_level)

total_map_plus_noise = CMB_T_convolved + Noise

## plot the result
p=cmb_modules.Plot_CMB_Map(total_map_plus_noise,c_min,c_max,X_width,Y_width)

## Apodize  the map to eliminate edge effects

Before doing any kind Fourier operation, we must apodize the maps to eliminate edges effects. Edge effects come about because the Fourier transform treats a square array as having periodic boundaries.  Thus if we take the Fourier transform of a 2-dimensional map and the values on the left and right side (and also, top and bottom) of the map don't match, we end up generating spurious signals.   In this example we use a cosine window to smoothly roll off the signal to zero as we approach the edges of the map.  There are many choices of windows that trade sensitivity loss, coupling of adjacent modes, and ringing.

In [ ]:
def cosine_window(N):
    "makes a cosine window for apodizing to avoid edges effects in the 2d FFT" 
    # make a 2d coordinate system
    N=int(N) 
    ones = np.ones(N)
    inds  = (((np.arange(N)+.5 - N/2.)/N) *np.pi) ## eg runs from -pi/2 to pi/2
    X = np.outer(ones,inds)
    Y = np.transpose(X)
  
    # make a window map
    window_map = np.cos(X) * np.cos(Y)
   
    # return the window map
    return(window_map)
  ###############################

window = (cosine_window(N))
    
apodized_map = window * total_map_plus_noise

p=cmb_modules.Plot_CMB_Map(apodized_map,c_min,c_max,X_width,Y_width)

This shows our simulated map with a cosine window applied to eliminate edge effects.  It is obvious from this map that we are suppressing the signal here.

<font color='red'>EXERCISE: </font>  There are an huge number of well studied windows with various combinations of properties.  Some minimize mode coupling, others minimize signal loss, while others maximize some combination of the two.  Using the [Wikipedia article Fourier transform windows](https://en.wikipedia.org/wiki/Window_function), choose one of your favorites and implement it as an option.  Compare the impact of the new window compared to the simple cosine window on the map.

In [52]:
## your code and plots go here

Your comments go here.

## Power spectra basics

Here's a quick reminder about power spectra (of course you can skip this part if you already know everything about power spectra). 

Let's start by looking at the maths on the full sky, later on we will only work on "flat sky" (i.e. a small enough part of the sky so that it can be approximated as flat), but the concepts are the same. Every map $\Theta(\hat{n})$ (for instance a scalar quantity on a sphere) can be decomposed into spherical harmonics:

$$\Theta(\hat{n}) = \sum_{\ell=0}^{+\infty} \sum_{m=-\ell}^{\ell} a_{\ell m} Y_{\ell m} (\hat{n}) $$

where $\hat{n}$ is the line of sight. The spherical harmonics coefficients $a_{\ell m}$ are the projection of our map $\Theta(\hat{n})$ on a given spherical harmonic $Y_{\ell m}$:
$$ a_{\ell m} = \int_{\hat{n}} d\Omega \ Y^*_{\ell m} (\hat{n})\ \Theta(\hat{n}) $$

You can think of it as a Fourier transform, but instead of the Fourier coefficients, a spherical harmonic transform gives $a_{\ell m}$ coefficients, while $Y_{\ell m} (\hat{n})$ plays the same role as sine or cosine in Fourier transforms. The multipole $\ell$ is linked to the angular scale probed by $Y_{\ell m}$, with $\theta \sim \frac{180\degree}{\ell}$.

Like Fourier transforms, SHTs are bijective, i.e. all pixelized maps are described by a unique set of $a_{\ell m}$, from which you can reconstruct the map. $a_{\ell, m}$ are imaginary numbers, but for a real-number map, one can show that $a^*_{\ell, m} = -a_{\ell, -m}$.

Below is a small bit of code that plots what different $Y_{\ell m} (\hat{n})$ look like on a sphere (you can change $\ell$ and $m$).

In [ ]:
! pip install healpy

In [ ]:
# Install healpy for this (pip install healpy should work on Google Colab)
import healpy as hp

# Choose any l, m (remember that -l <= m <= l)
l, m = 17, 4

NSIDE = 64
LMAX = 2*NSIDE

# Generates an empty map and add the wanted alm
zeros_map = np.zeros(hp.nside2npix(NSIDE))
alms  = hp.map2alm(zeros_map, lmax=LMAX)
m_sign = 1 if m ==0 else m / abs(m)
alms[hp.Alm.getidx(LMAX, l, abs(m))] = .5 * m_sign
ylm = hp.alm2map(alms, nside=NSIDE)

hp.mollview(ylm, title=None, cbar=None, cmap="planck", min=-1, max=1)

The CMB is an isotropic [__Gaussian Random Field__](https://en.wikipedia.org/wiki/Gaussian_random_field). For our purposes, it means that its $a_{\ell, m}$ are independent random gaussian variables, with mean 0 and variance given by the __angular power spectrum__:

$$ a_{\ell m} \sim \mathcal{N} (0, C_\ell) $$

Hence, all the statistical information of a gaussian field is entirely contained in its power spectrum (and as you saw in previous courses, these statistical properties are predicted by theory !).

To estimate the $\hat{C}_\ell$ of a given map, we can measure the variance of its $a_{\ell, m}$ :


$$ \hat{C}_\ell = \frac{1}{2\ell + 1} \sum_m a^{*}_{\ell m} a_{\ell m} = \frac{1}{2\ell + 1} \sum_m ||a_{\ell m}||^2 $$

note that you can also define a cross-power spectrum as:

$$ \hat{C}^{XY}_\ell = \frac{1}{2\ell + 1} \sum_m a^{X,*}_{\ell m} a^Y_{\ell m} $$

The relation between $C_\ell$ and $a_{\ell m}$ (and so maps) is not one-by-one, multiple $a_{\ell m}$ distributions can have the same variance and hence the same $C_\ell$. To get a map or a set of $a_{\ell,m}$ from a given power spectrum, you need to __draw__ a set of $a_{\ell, m}$ from the gaussian distrubtion given by the power spectrum $C_\ell$. This also why we now make the difference between the "true" underlying $C_\ell$, and the _estimated_ $\hat{C_\ell}$, that we measure from a set of $a_{\ell, m}$. (Note that we also often use $D_\ell = \frac{\ell(\ell + 1)}{2\pi} C_\ell$).

The _ensemble_ average, i.e. the average over all realization, is denoted with $\langle \dots \rangle$.

The fact that we estimate $\hat{C}_\ell$ from the variance of a single $a_{\ell, m}$ realization is actually very important. For a signal-only map, on average, $\langle \hat{C}_\ell \rangle = C_\ell $, the estimator is not biaised. But especially for low multipoles $\ell$, we don't have an infinite number of modes $m$ to measure the variance of $a_{\ell, m}$ on, so $\hat{C}_\ell$ is itself "noisy", even for a signal-only map ! The is the __cosmic variance__: since we can only observe one realization of the CMB, we can only measure one set of $a_{\ell, m}$, so even without any instrumental effects, $\hat{C}_\ell$ is not the exact input $C_\ell$ ! One can show that the variance on a given $\hat{C}_\ell$ is :

$$ Var(\hat{C}_\ell) = \frac{2}{2\ell +1} C_\ell^2 $$

and for a cross spectrum :

$$ Var(\hat{C}^{XY}_\ell) = \frac{1}{2\ell +1} ((C^{XY}_\ell)^2 + C^{XX}_\ell C^{YY}_\ell) \tag{1} $$

As you can see, this variance is higher for lower multipoles, where there are less $m$ modes to measure.

## Naive Powerspectrum

Now we'll measure the power spectrum of the different maps we've made so far. In flat sky, the power spectrum is computed by: (1) applying a 2d FFT, (2) taking the absolute value squared of this map in Fourier space ($k_x$ and $k_y$), and (3) averaging the signal in annular bins of $k = \sqrt{k_x^2 + k_y^2}$.  These bins are converted to bins in $\ell$ with the scaling: $\ell = k* 2 \pi$ per the flat sky approximation.   NOTE: step 3 (averaging in radial bins) is how we convert our 2d Fourier map into a 1d power spectrum.

Our spectrum code takes two maps as inputs to allow for cross spectra.

In [ ]:
#### parameters for setting up the spectrum
delta_ell = 50.
# ell_min = 100.
ell_max = 5000.

if max(ell)< ell_max: 
    print('WARNING: Your theory curves end before the binned ell_max')

def calculate_2d_spectrum(Map1,Map2,delta_ell,ell_max,pix_size,N):
    "calcualtes the power spectrum of a 2d map by FFTing, squaring, and azimuthally averaging"
    ell_min = 0
    N=int(N)
    # make a 2d ell coordinate system
    ones = np.ones(N)
    inds  = (np.arange(N)+.5 - N/2.) /(N-1.)
    kX = np.outer(ones,inds) / (pix_size/60. * np.pi/180.)
    kY = np.transpose(kX)
    K = np.sqrt(kX**2. + kY**2.)
    ell_scale_factor = 2. * np.pi 
    ell2d = K * ell_scale_factor
    
    # make an array to hold the power spectrum results
    N_bins = int((ell_max-ell_min)/delta_ell)
    ell_array = np.arange(N_bins)
    CL_array = np.zeros(N_bins)
    
    # get the 2d fourier transform of the map
    FMap1 = np.fft.ifft2(np.fft.fftshift(Map1))
    FMap2 = np.fft.ifft2(np.fft.fftshift(Map2))
    PSMap = np.fft.fftshift(np.real(np.conj(FMap1) * FMap2))
    # fill out the spectra

    for i in range(N_bins):
        ell_array[i] = (i + 0.5) * delta_ell + ell_min
        inds_in_bin = ((ell2d >= (i*delta_ell  + ell_min)) * (ell2d < ((i+1)* delta_ell + ell_min))).nonzero()
        CL_array[i] = np.mean(PSMap[inds_in_bin])

    # return the power spectrum and ell bins
    return(ell_array,CL_array*(pix_size /60.* np.pi/180.) /4)

## make a power spectrum
binned_ell, binned_spectrum = calculate_2d_spectrum(apodized_map,apodized_map,delta_ell,ell_max,pix_size,N)

# to go from Cl to Dl
fac_binned = binned_ell * (binned_ell+1.) /2. / np.pi

fig, ax = plt.subplots(figsize=(7, 4))

ax.plot(binned_ell,binned_spectrum* fac_binned, label=r"Measured $\hat{C}_\ell$")
ax.plot(ell,DlTT, label=r"Input $C_\ell$", color="black")
ax.set_ylabel(r'$D_{\ell}$ [$\mu$K$^2$]', fontsize=18)
ax.set_xlabel(r'$\ell$', fontsize=18)
ax.set_yscale("log")
ax.legend(fontsize=12)

This plot shows the input CMB power spectrum (black) and the naive power spectrum we estimated from our CMB map (blue).  You can see that the measured power spectrum does not match the input, in the next sections we will see how to correct this naive power spectrum for different effects in order to recover a clean CMB power spectrum.

<font color='red'>EXERCISE: </font>  Looking at the difference between the blue and black lines, do you have any idea of what could explain their discrepancy ? 

Your comments go here.

## Effect of masking

The first effect that we'll study is the effect of masking. Since we're only looking at a part of the sky (and the apodization masks a part of our already small patch), it's completely expected that we don't recover the same amplitude as the theory power spectrum. At first order, the lost power is proportionnal to the sky fraction $f_{sky}$.

$$ \langle \hat{C}^{masked}_\ell \rangle = f_{sky} C_\ell $$

This also increases the variance, since there are also much less modes to measure:

$$ Var(\hat{C}_\ell) = \frac{1}{f_{sky}} \frac{2}{2\ell +1} C_\ell^2 $$


Masking the sky also induces [mode-coupling](#Mode-coupling-matrix), especially for small sky fractions or weirdly shaped masks, which is especially problematic for polarization. We won't have time to go into details about this, and it will not matter much for our applications. This part will be studied in the __Mode coupling__ part of the course later on.

Here is a function that lets you compute the sky fraction $f_{sky}$ from characteristics of you patch and apodization window. If no apodization was used, leave `window=` blank or put it to `1`. At first order, the $f_{sky}$ is just the fraction of signal masked by your window (so something like `np.mean(window)`), but the effective number of masked modes is actually a bit different.

In [ ]:
def get_fsky(pix_size, N, window=1):
    patch_area = (pix_size / 60 * np.pi / 180)**2 * N**2
    window_fsky = np.mean(window**2)
    return patch_area  / (4 * np.pi) * window_fsky

print(f"fsky without window = {get_fsky(pix_size, N):.2%}")
print(f"fsky with window = {get_fsky(pix_size, N, window):.2%}")

Our patch only covers 0.18% of the sky ! That's why the power spectrum we got was so small compared to the theory.

<font color='red'>EXERCISE: </font> Correct the measured powr spectrum from this effect. You can also check that it works for different apodization windows.

You should see that this corrects the amplitude effect that we saw, but there still some discrepancy that we will study now.

## Beam transfer function

Let's now take a look at the effect of the beam on the power spectrum. For an isotropic (same accross the whole sky) gaussian beam, the beam transfer function is :

$$ b_\ell = \exp(-\frac{\ell (\ell + 1)}{2} \frac{\theta_{\mathrm{FWHM}}}{8 \log 2}) $$

where $\theta_{\mathrm{FWHM}}$ is the Full Width at Half Max of the gaussian beam. The beam transfer function usually start at 1 for low multipoles (large scales), then drops to zero for scales much smaller than the beam size (these scales are completely smoothed out). Hence, beam smearing results in a loss of power on high multipoles (small scales). The smaller the beam size $\mathrm{FWHM}$ is, the smaller this loss of power will be, and higher multipoles will be accessible.

The effect on power spectrum is :
$$ C^{obs}_\ell  = b_\ell^2 C_\ell $$

<font color='red'>EXERCISE: </font> Correct the power spectrum by this effect.

### Calibrating the noise bias

The noise biais is the noise contribution to the power spectrum :
$$ \langle \hat{C}_\ell \rangle = C_\ell + N_\ell $$

We'll se two ways to remove this biais: Monte Carlo and cross-spectrum.

#### Noise bias using Monte Carlo approach

If you know the properties of the noise of your experiment and you can make realizations of it, you can simulate a bunch of these and measure the noise power spectrum on it.

In [ ]:
N_iterations = 16

# This function takes multiple spectra and averages them
def average_N_spectra(spectra):
    avgSpectra = np.mean(spectra, axis=0)
    rmsSpectra = np.std(spectra, axis=0)
    return avgSpectra, rmsSpectra


noise_only  = np.zeros([N_iterations,int(ell_max/delta_ell)])
for i in range(N_iterations):
    Noise = cmb_modules.make_noise_map(N,pix_size,white_noise_level,atmospheric_noise_level,one_over_f_noise_level)
    binned_ell_cur, binned_spectrum_cur = calculate_2d_spectrum(Noise*window,Noise*window,delta_ell,ell_max,pix_size,N)
    noise_only[i,:] = binned_spectrum_cur
    sys.stdout.write("\r noise only sims, iterations complete: %d of %d" % ((i+1),N_iterations) )
    sys.stdout.flush()

noise_only_mean_spectrum, _ = average_N_spectra(noise_only)

fig, ax = plt.subplots(figsize=(7, 4))

ax.plot(ell,DlTT, label=r"Input $C_\ell$", color="black")
ax.plot(binned_ell,binned_spectrum* fac_binned / get_fsky(pix_size, N, window=window), label=r"Measured $\hat{C}_\ell + N_\ell$")
ax.plot(binned_ell,noise_only_mean_spectrum* fac_binned / get_fsky(pix_size, N, window=window), label=r"Monte Carlo $N_\ell$")
ax.plot(binned_ell,(binned_spectrum - noise_only_mean_spectrum)* fac_binned / get_fsky(pix_size, N, window=window), label=r"Corrected $\hat{C}_\ell$")

ax.set_ylabel(r'$D_{\ell}$ [$\mu$K$^2$]', fontsize=18)
ax.set_xlabel(r'$\ell$', fontsize=18)
ax.set_yscale("log")
ax.legend(fontsize=12)

The green curve in this plot shows our unbiased estimate for the spectrum.  The orange curve shows our estimate for the noise only additive bias (you can use it for the rest of the notebook). 

<font color='red'>EXERCISE: </font>  What could go wrong with this approach ? You can test your hypotheses by trying to recover the CMB from CMB+noise maps.

your comments go here

#### Noise biais using cross spectrum

The most used approach for modern CMB experiments is the cross spectrum approach. If you can divide your data into multiple  independent maps, you can then do a cross spectrum between these. The result will be that you only keep the "signal" part (that is the same between the different data splits), while is the "noise" part is completely independent for the two splits, will vanish in average:

$$ \langle \hat{C}^{XY}_\ell\rangle = \langle\frac{1}{2\ell + 1} \sum_m a^{X, *}_{\ell m} a^{Y}_{\ell m} \rangle 
=  \langle \frac{1}{2\ell + 1} \sum_m \left(a^{signal,X}_{\ell m} + a^{noise,X}_{\ell m}\right)^*  \left(a^{signal,Y}_{\ell m} + a^{noise,Y}_{\ell m}\right) \rangle 
\\= \langle \frac{1}{2\ell + 1} \sum_m \left( a^{signal,X}_{\ell m} a^{signal,Y}_{\ell m} + a^{noise,X}_{\ell m}a^{noise,Y}_{\ell m} + a^{signal,X}_{\ell m}a^{noise,Y}_{\ell m} + a^{noise,X}_{\ell m}a^{signal,Y}_{\ell m} \right) \rangle
= C^{signal}_\ell $$

Where we have used the fact that both splits see the same signal, but have independent noise. Of course, everything the two splits have in common is considered "signal" here, so the splits have to be done with great care. For instance, ground experiments have a noise contribution from the ground pick-up (light of nearby mointains for instance) that can be the same across multiple data splits.

This results in some information loss (since you are throwing out the auto-correlation of each subset), but in theory it completely eliminates potential measurement bias from an incorrect noise model. Increasing the number of splits reduces the information lost (for instance ACT used 4 splits and SPT used 30).

<font color='red'>EXERCISE: </font>  Make 2 maps with the same signal realization and two different noise realization, and compare the power spectra of individual maps and the cross spectrum. Use it to get an estimate of the noise biais, and compare it to what we got with the Monte Carlo approach.

In [57]:
# Your codes goes here

### Removing the foregrounds

Foregrounds can either be removed on the maps or at the power spectrum level. In both cases, the main idea is to use the fact that foregrounds have a very different frequency dependence that the CMB, which allows to do component separation. 

CMB experiments use foreground templates coming from simulations, that are fitted jointly with the cosmology as nuisance parameters (even though some people find more interest into forregrounds than cosmology !).

Let's take a look at the power spectrum of the different foregrounds we've seen yet.

In [ ]:
PSMap_poisson = cmb_modules.Poisson_source_component(N,pix_size,Number_of_Sources,Amplitude_of_Sources) 
PSMap_exp =  cmb_modules.Exponential_source_component(N,pix_size,Number_of_Sources_EX,Amplitude_of_Sources_EX)

SZMap,_ = cmb_modules.SZ_source_component(N,pix_size,Number_of_SZ_Clusters,Mean_Amplitude_of_SZ_Clusters,SZ_beta,SZ_Theta_core,False)

binned_ell, binned_spectrum_poisson = calculate_2d_spectrum(PSMap_poisson*window,PSMap_poisson*window,delta_ell,ell_max,pix_size,N)
binned_ell, binned_spectrum_exp = calculate_2d_spectrum(PSMap_exp*window,PSMap_exp*window,delta_ell,ell_max,pix_size,N)
binned_ell, binned_spectrum_SZ = calculate_2d_spectrum(SZMap*window,SZMap*window,delta_ell,ell_max,pix_size,N)

# to go from Cl to Dl
fac_binned = binned_ell * (binned_ell+1.) /2. / np.pi
fsky = get_fsky(pix_size, N, window)

fig, ax = plt.subplots(figsize=(7, 4))

ax.plot(binned_ell,binned_spectrum_poisson* fac_binned / fsky, label=r"Poisson ")
ax.plot(binned_ell,binned_spectrum_exp* fac_binned / fsky, label=r"Exponential ")
ax.plot(binned_ell,binned_spectrum_SZ* fac_binned / fsky, label=r"SZ ")
ax.plot(ell,DlTT, label=r"CMB $C_\ell$", color="black")
ax.set_ylabel(r'$D_{\ell}$ [$\mu$K$^2$]', fontsize=18)
ax.set_xlabel(r'$\ell$', fontsize=18)
ax.set_yscale("log")
ax.legend(fontsize=12)
ax.set_ylim(1e-3, 1e4)

Here, we only have one map, se we can't do component separation, but we can still use a template for the "shape" of the power spectrum and try to fit it to our spectrum. 



<font color='red'>EXERCISE: </font>  The "exponential Poisson" foreground component can be modeled as $C_\ell \sim \ell^2$. Create a CMB + exp. Poisson map (+ instrumental effects if you want), compute the power spectrum and try to fit this model to the high-$\ell$ of its power spectrum, where the foregrounds are dominant. Check that you recover the right power spectrum by measuring the power spectrum of the foreground-only map.

### Quantifying the error bars

We now have a power spectrum that makes sense and that we know what it's made of ! Let's now try to put error bars on it.

The error bars can be estimated by the Monte Carlo approach, i.e. drawing a bunch of simulations and measuring the RMS of their power spectrum :

In [ ]:
SplusN  = np.zeros([N_iterations,int(ell_max/delta_ell)])

for i in range(N_iterations):
    CMB_T = cmb_modules.make_CMB_T_map(N,pix_size,ell,DlTT)

    PSMap = cmb_modules.Poisson_source_component(N,pix_size,Number_of_Sources,Amplitude_of_Sources) 
    PSMap += cmb_modules.Exponential_source_component(N,pix_size,Number_of_Sources_EX,Amplitude_of_Sources_EX)
    SZMap,_ = cmb_modules.SZ_source_component(N,pix_size,Number_of_SZ_Clusters,\
                                      Mean_Amplitude_of_SZ_Clusters,SZ_beta,SZ_Theta_core,False)
    
    CMB_T  = CMB_T + PSMap + SZMap

    CMB_T_convolved = cmb_modules.convolve_map_with_gaussian_beam(N,pix_size,beam_size_fwhp,CMB_T)
    Noise = cmb_modules.make_noise_map(N,pix_size,white_noise_level,atmospheric_noise_level,one_over_f_noise_level)
    binned_ell_cur, binned_spectrum_cur = calculate_2d_spectrum((CMB_T_convolved+Noise)*window\
                                                                ,(CMB_T_convolved+Noise)*window\
                                                                ,delta_ell,ell_max,pix_size,N)
    SplusN[i,:] = binned_spectrum_cur
    sys.stdout.write("\r signal and noise sims, iterations complete: %d of %d" % ((i+1),N_iterations) )
    sys.stdout.flush()

_,rms_sig_plus_noise = average_N_spectra(SplusN)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

ax.errorbar(
    binned_ell, 
    (binned_spectrum - noise_only_mean_spectrum) * fac_binned / fsky,
    yerr=rms_sig_plus_noise* fac_binned / fsky,
    elinewidth=2
)
ax.semilogy(ell,DlTT,color='r')
ax.set_xlabel(r"$\ell$", fontsize=18)
ax.set_ylabel(r"$D_\ell$", fontsize=18)

And there you have it.  That is how you compute a CMB power spectrum and error bars.   If you want to fit cosmology to these data you can re-run CAMB varying cosmological parameters and compute the likelihood difference between these models and the data.

<font color='red'>EXERCISE: </font>: Compare the Monte Carlo errorbars with the analytical variance of a gaussian field given earlier. Try with/without foregrounds and comment.

## Additionnal content

### Mode coupling

The effect of the mask is actually more that just a lost of power. For this part, we'll work in full-sky since it's way easier to highlight this effect.

Since spherical harmonics $Y_{\ell, m}$ are defined on the full sky, when a masked signal is projected on the spherical harmonics basis, we actually get a biaised estimation of the $a_{\ell, m}$. This will especially couple close multipoles $\ell$.

To see this effect, we create a map from a power spectrum with only one multipole different from zero: for instance $C_\ell = \delta_{\ell=50}$. We then mask this map, and compare its power spectrum to the input.

In [ ]:
LMAX = 1000
NSIDE = 512

multipole = 50
deg_around_galplane = 25
mask_apod_deg = 5

def apod_healpix_mask(mask, apod_deg):
    dist_map = hp.dist2holes(mask)
    apod_mask = .5 - .5* np.cos(-180 * dist_map / apod_deg)
    apod_mask[dist_map > np.pi / 180 * apod_deg] = 1
    return apod_mask

gal_mask = np.ones(hp.nside2npix(512))
gal_mask[hp.query_strip(512, (90+deg_around_galplane) * np.pi / 180, (90-deg_around_galplane) * np.pi / 180)] = 0
gal_mask = apod_healpix_mask(gal_mask, mask_apod_deg)

C_dll = np.zeros(LMAX)
C_dll[multipole] = 1.

healpix_map = hp.synfast(C_dll, nside=NSIDE)
hp.mollview(healpix_map * gal_mask, min=-10, max=10, title=None, cbar=None, cmap="planck")

C_ell_est = hp.anafast(healpix_map, lmax=LMAX-1)
C_ell_est_masked = hp.anafast(healpix_map*gal_mask, lmax=LMAX-1)
ls = np.arange(LMAX)
fac=  ls * (ls + 1) / (2 * np.pi)

fig, ax = plt.subplots(figsize=(7, 4))

ax.plot(ls, C_dll * fac, label=r"Input $C_\ell$", color="black")
ax.plot(ls, C_ell_est * fac, label=r"Full sky $\hat{C}_\ell$")
ax.plot(ls, C_ell_est_masked * fac / np.mean(gal_mask), label=r"Masked $\hat{C}_\ell$")
ax.set_xlabel(r"$\ell$", fontsize=18)
ax.set_ylabel(r"$D_\ell$", fontsize=18)
ax.legend(fontsize=14)
ax.set_xlim(multipole-10, multipole+10)

For a full sky $\hat{C}_\ell$, you recover the exact same spectrum (plus or minus the cosmic variance), while for a $\hat{C}_\ell$ measured on a masked map, the signal "leaks" in nearby multipoles.

This is important to keep in mind, especially for very small or weirdly shaped masks. This effect can also be corrected, but this is out of the scope of this course (you can find more details in [this paper](https://arxiv.org/abs/astro-ph/0105302v1)).

<font color='red'>EXERCISE: </font> Can you think about simple tricks to drastically reduce the impact of this effect ? Try to apply it. (Hint: Mode coupling only affect multipoles very close to each other.)

### Pixel window function

When making a map with data coming from scans of the sky by a telescope (see course about mapmaking), you need to "bin" data points into pixels of your final maps. This results in convolving a __pixel window__ to the true underlying map. In the spectrum space, convolution become products, this results as a multiplicative biais, especially important at small scales. 

For Nearest-Neighbor pixel window (where the signal of time-ordered data is given to the nearest pixel), the pixel window function is a rectangular function, so the pixel window is a sinc function:

$$ P_\ell = \mathrm{sinc} \left(\frac{\ell \theta_{pixel}}{2\pi} \right)$$

To correct an observed power spectrum from this effect, you need to divide $C_\ell^{obs}$ by $P_\ell^2$.

Note that for a simulated map, the signal of each pixel is already the signal of the middle of the pixel, so you don't need to correct for this effect.

